In [1]:
import time
from collections import namedtuple, defaultdict
import pathlib

from tqdm.notebook import tqdm
import numpy as np
import pandas as pd

from sdrsdm import TriadicMemory

# Setup

In [2]:
RNG = np.random.default_rng(100)
N = 500
P = 5

In [16]:
def my_empty_sdr(n=N, p=P):
    return np.array([], dtype=int)

def my_random_sdr(n=N, p=P):
    inds = RNG.choice(n, p, replace=False)
    inds.sort()
    return inds

def my_sdr_union(s1, s2):
    united = np.r_[s1, s2]
    return np.unique(united)

def my_sdr_overlap(s1, s2):
    s1 = set(s1)
    s2 = set(s2)
    return len(s1 & s2)

def my_sdr_is_same(s1, s2, p=P):
    return my_sdr_overlap(s1, s2) >= p

def my_sdr_is_equal(s1, s2):
    if len(s1) != len(s2):
        return False

    return np.all(s1 == s2)

In [6]:
class Codec():
    def __init__(self, n, p):
        self.n = n
        self.p = p
        self.str_to_sdr = {}
        self.sdr_to_str = {}

    def clear(self):
        self.str_to_sdr = {}
        self.sdr_to_str = {}

    def __call__(self, x):
        if x is None:
            return my_empty_sdr(self.n, self.p)
    
        if type(x) == np.ndarray:
            if len(x) == 0:
                return None
                
            key = tuple(map(int, x))
            return self.sdr_to_str.get(key, x)
        elif type(x) == str:
            if x in self.str_to_sdr:
                return self.str_to_sdr[x]
            else:
                new_sdr = my_random_sdr(self.n, self.p)
                self.str_to_sdr[x] = new_sdr
                self.sdr_to_str[tuple(map(int, new_sdr))] = x
                return new_sdr
        else:
            assert False, f'Unknown type "{type(x)}"'

In [7]:
codec = Codec(N, P)
strs = ['a', 'b', 'c', 'd', 'e']
sdrs = list(map(codec, strs))

for sdr, s in zip(sdrs, strs):
    assert codec(sdr) == s

r = my_random_sdr()
assert np.all(codec(r) == r)
assert len(codec(None)) == 0
assert codec(np.array([])) is None

# How contexts are resolved?

In [21]:
codec = Codec(N, P)

M1 = TriadicMemory(N, P)
M2 = TriadicMemory(N, P)
y, c, u, v, prediction = (my_empty_sdr(),) * 5
clist = []
cset = set()

test_input = 'ABCABC'
history = ''
columns = defaultdict(list)

for inp, next_inp in zip(test_input, test_input[1:] + '$'):
    x = my_sdr_union(y, c)
    y = codec(inp)
    
    if not my_sdr_is_same(prediction, y):
        M2.store(u, v, y)
    
    c = M1.query_Z(x, y)
    x_hat = M1.query_X(y, c)
    
    if my_sdr_overlap(x_hat, x) < P:
        c = my_random_sdr()
        M1.store(x, y, c)
    
    u = x
    v = y
    prediction = M2.query_Z(u, v)
    
    if not tuple(c) in cset:
        cset.add(tuple(c))
        clist.append(c)

    columns['history'].append(history)
    columns['input'].append(inp)
    columns['next_pred'].append(codec(prediction))
    columns['next_true'].append(next_inp)
    columns['is_correct_pred'].append(int(my_sdr_is_equal(codec(next_inp), prediction)))
    history = history + inp

pd.DataFrame(columns)

,history,input,next_pred,next_true,is_correct_pred
0,,A,None,B,0
1,A,B,None,C,0
2,AB,C,None,A,0
3,ABC,A,None,B,0
4,ABCA,B,C,C,1
5,ABCAB,C,A,$,0


In [22]:
a, b = M1.query_Z(codec('A'), codec('B')), clist[1]
assert np.all(a == b)
a, b

(array([ 42, 221, 282, 305, 499]), array([ 42, 221, 282, 305, 499]))

In [23]:
a, b = M1.query_Z(clist[0], codec('B')), clist[1]
assert np.all(a == b)
a, b

(array([ 42, 221, 282, 305, 499]), array([ 42, 221, 282, 305, 499]))

В M1 контексты сохраняются с ключами: x = prev_context | prev_input, y = input. А раз так, то контекст можно разрезловить по частичному совпадению ключей, например:
1) x = prev_input, y = input
2) x = prev_context, y = input   

In [24]:
x1 = my_sdr_union(codec('A'), my_sdr_union(my_random_sdr(), my_random_sdr()))
y1 = my_sdr_union(codec('B'), my_sdr_union(my_random_sdr(), my_random_sdr()))
a, b = M1.query_Z(x1, y1), clist[1]
assert np.all(a == b)
a, b

(array([ 42, 221, 282, 305, 499]), array([ 42, 221, 282, 305, 499]))

Видно, что добавление лишнего шума (по два СДР для x1 и y1), всё равно позволяют корректно разрешить контекст 'AB'.

In [25]:
codec(M2.query_Z(codec('A'), codec('B')))

'C'

Даже без указания контекста M2 для AB возвращает C

In [39]:
# добавим в M2 Z для AB, т.е. AB -> Z
M2.store(codec('A'), codec('B'), codec('Z'))
a = codec(M2.query_Z(codec('A'), codec('B')))
assert my_sdr_is_same(codec('C'), a)
assert my_sdr_is_same(codec('Z'), a)
a

array([ 17,  31,  44,  91, 250, 253, 268, 396, 423, 468])

после добавления 'Z' в M2 последний будет возвращать суперпозицию C и Z в ответ на запрос query_Z('A', 'B')

In [35]:
a = codec(M2.query_Z(my_sdr_union(codec('A'), clist[0]), codec('B')))
assert a == 'C'
a

'C'

но, если указать контекст clist[0], то M2 будет возвращать C. То есть без дополнительной подсказки возвращается всё, что знаем (суперпозиция), с подсказкой - уже конкретное

# Temporal memory

In [17]:
class TemporalContextMemory():
    def __init__(self, N, P):
        self.n = N
        self.p = P
        self.mem = TriadicMemory(N, P)
        self.prev_inp, self.context = (my_empty_sdr(self.n, self.p),) * 2
        self.context_usage_stats = {}

    def flush(self):
        self.prev_inp, self.context = (my_empty_sdr(self.n, self.p),) * 2
        return self.context

    def __call__(self, inp):
        assert len(inp) > 0
        x = my_sdr_union(self.prev_inp, self.context)
        self.context = self.mem.query_Z(x, inp)
        x_cleanup = self.mem.query_X(inp, self.context) # cleanup via roundtrip
        
        if not my_sdr_is_same(x_cleanup, x, self.p):
            self.context = my_random_sdr(self.n, self.p)
            self.mem.store(x, inp, self.context)
            self.context_usage_stats[tuple(map(int, self.context))] = 1
        else:
            key = tuple(map(int, self.context))
            assert key in self.context_usage_stats
            self.context_usage_stats[key] += 1
            
        self.prev_inp = inp
        return self.context

def create_context_mem():
    mem = TriadicMemory(N, P)
    prev_inp, context = (my_empty_sdr(),) * 2

    def process(inp):
        nonlocal prev_inp, context
        
        if len(inp) == 0:
            # Flush
            prev_inp, context = (my_empty_sdr(),) * 2
            return context
        
        x = my_sdr_union(prev_inp, context)
        context = mem.query_Z(x, inp)
        x_cleanup = mem.query_X(inp, context) # cleanup via roundtrip
        
        if my_sdr_overlap(x_cleanup, x) < P:
            context = my_random_sdr()
            mem.store(x, inp, context)

        prev_inp = inp
        return context

    return process

In [19]:
# context_mem = create_context_mem()
context_mem = TemporalContextMemory(N, P)
mem = TriadicMemory(N, P)
prev_inp, u, v, prediction = (my_empty_sdr(),) * 4

test_input = 'banana' * 5
history = ''
columns = defaultdict(list)

for inp, next_inp in zip(test_input, test_input[1:] + '$'):
    inp_orig = inp
    inp = codec(inp)
    context = context_mem(inp)

    if not my_sdr_is_same(prediction, inp):
        mem.store(u, v, inp)

    u, v = my_sdr_union(prev_inp, context), inp
    prediction = mem.query_Z(u, v)
    prev_inp = inp
    
    columns['history'].append(history)
    columns['input'].append(inp_orig)
    columns['next_pred'].append(codec(prediction))
    columns['next_true'].append(next_inp)
    columns['is_correct_pred'].append(int(my_sdr_is_equal(codec(next_inp), prediction)))
    history = history + inp_orig

pd.DataFrame(columns)

,history,input,next_pred,next_true,is_correct_pred
0,,b,None,a,0
1,b,a,None,n,0
2,ba,n,None,a,0
3,ban,a,None,n,0
4,bana,n,a,a,1
5,banan,a,n,b,0
6,banana,b,None,a,0
7,bananab,a,n,n,1
8,bananaba,n,a,a,1
9,bananaban,a,"[40, 70, 136, 142, 304, 328, 341, 412, 419, 453]",n,0


Результаты (по колонке is_correct_pred) соответствуют тем, что в ноутбукe "Temporal Memory Elementary Algorithm.nb"

![](./img/temporal-mathematica-banana.jpg)

# Deep temporal memory

In [203]:
# R_list = list(map(lambda _: create_context_mem(), range(7)))
R_list = list(map(lambda _: TemporalContextMemory(N, P), range(7)))
mem = TriadicMemory(N, P)
prev_inp, context, x, y, prediction = (my_empty_sdr(),) * 5

test_input = 'banana' * 5
history = ''
columns = defaultdict(list)

for inp, next_inp in zip(test_input, test_input[1:] + '$'):
    inp_hat = codec(inp)
    t_list = []
    
    for R in R_list:
        t = t_list[-1] if t_list else inp_hat
        t_list.append(R(t))

    if not my_sdr_is_same(prediction, inp_hat):
        mem.store(x, y, inp_hat)

    x = my_sdr_union(t_list[0], t_list[3])
    y = my_sdr_union(t_list[1], t_list[6])
    prediction = mem.query_Z(x, y)

    columns['history'].append(history)
    columns['input'].append(inp)
    columns['next_pred'].append(codec(prediction))
    columns['next_true'].append(next_inp)
    columns['is_correct_pred'].append(int(my_sdr_is_same(codec(next_inp), prediction)))
    history = history + inp

pd.DataFrame(columns)

,history,input,next_pred,next_true,is_correct_pred
0,,b,None,a,0
1,b,a,None,n,0
2,ba,n,None,a,0
3,ban,a,None,n,0
4,bana,n,None,a,0
5,banan,a,n,b,0
6,banana,b,None,a,0
7,bananab,a,None,n,0
8,bananaba,n,a,a,1
9,bananaban,a,"[51, 183, 205, 259, 297, 343, 395, 470, 482, 494]",n,0


Результаты (по колонке is_correct_pred) соответствуют тем, что в ноутбукe "Temporal Memory Elementary Algorithm.nb"

![](./img/temporal-mathematica-banana-2.jpg)

# Consume and generate

In [6]:
def preprocess_text(text):
    words = text.split()
    words = list(map(lambda t: t.strip('.,:?!-—'), words))
    return words

In [7]:
text = pathlib.Path('ulysses.txt').read_text(encoding='utf-8-sig')
words = preprocess_text(text)[:50000]

In [8]:
N = 1000
codec(...) # clear cache

In [9]:
M0 = TriadicMemory(N, P)
M1 = TriadicMemory(N, P)
M2 = TriadicMemory(N, P)

In [10]:
prev_token, prev_bigram, prev_bigram_with_context, context, prediction = (my_empty_sdr(),) * 5
i, j, y, c, u, v, prediction = (my_empty_sdr(),) * 7

for run in range(4):
    hits, misses = 0, 0
    
    for word_ind, word in tqdm(enumerate(words), total=len(words)):
        token = codec(word)

        if False:
            j = i
            i = token
            bigram = M0.query_Z(i, j)

            if my_sdr_overlap(M0.query_Y(i, bigram), j) < P:
                bigram = my_random_sdr()
                M0.store(i, j, bigram)

            x = my_sdr_union(y, c)
            y = bigram

            c = M1.query_Z(x, y)

            if my_sdr_overlap(M1.query_X(y, c), x) < P:
                c = my_random_sdr()
                M1.store(x, y, c)

            if my_sdr_overlap(prediction, i) < P:
                M2.store(u, v, i)
                misses += 1
            else:
                hits += 1

            u = x
            v = y
            prediction = M2.query_Z(u, v)
        else:
            # Bigram
            bigram = M0.query_Z(token, prev_token)
            prev_token_cleanup = M0.query_Y(token, bigram)
    
            if my_sdr_overlap(prev_token, prev_token_cleanup) < P:
                bigram = my_random_sdr()
                M0.store(token, prev_token, bigram)
    
            # Context
            x = my_sdr_union(prev_bigram, context)
            context = M1.query_Z(x, bigram)
            x_cleanup = M1.query_X(bigram, context)
    
            if my_sdr_overlap(x, x_cleanup) < P:
                context = my_random_sdr()
                M1.store(x, bigram, context)
    
            # Prediction
            if my_sdr_overlap(prediction, token) < P:
                M2.store(prev_bigram_with_context, prev_bigram, token)
                misses += 1
            else:
                hits += 1
        
            prediction = M2.query_Z(x, bigram)
            prev_token = token
            prev_bigram = bigram
            prev_bigram_with_context = x
    
    print(f'{hits / (hits + misses):.4f}, hits={hits}, misses={misses}')

  0%|          | 0/50000 [00:00<?, ?it/s]

0.0257, hits=1285, misses=48715


  0%|          | 0/50000 [00:00<?, ?it/s]

0.7622, hits=38112, misses=11888


  0%|          | 0/50000 [00:00<?, ?it/s]

0.9307, hits=46533, misses=3467


  0%|          | 0/50000 [00:00<?, ?it/s]

0.9867, hits=49337, misses=663


In [12]:
# prompt = preprocess_text("London is a captial of great britain")
prompt = preprocess_text("I would like to go to the bar")
# prompt = words[:50]
prev_token, prev_bigram, prev_bigram_with_context, context, prediction = (my_empty_sdr(),) * 5

for word in prompt + [None] * 20:
    if word is None:
        # Prompt is over, now let's see what system generates in response to consumed prompt
        token = prediction

        if len(token) > P:
            for sdr, s in codec_cache.sdr_to_str.items():
                if my_sdr_overlap(sdr, token) == P:
                    word = s
                    token = sdr
                    break
        else:
            word = codec(token)
        
        print(" ", word)
    else:
        print("*", word)
        token = codec(word)
        
    # Bigram
    bigram = M0.query_Z(token, prev_token)
    prev_token_cleanup = M0.query_Y(token, bigram)

    if my_sdr_overlap(prev_token, prev_token_cleanup) < P:
        bigram = my_random_sdr()
        M0.store(token, prev_token, bigram)

    # Context
    x = my_sdr_union(prev_bigram, context)
    context = M1.query_Z(x, bigram)
    x_cleanup = M1.query_X(bigram, context)

    if my_sdr_overlap(x, x_cleanup) < P:
        context = my_random_sdr()
        M1.store(x, bigram, context)

    # Prediction
    # if my_sdr_overlap(prediction, token) < P:
    #     M2.store(prev_bigram_with_context, prev_bigram, token)

    prediction = M2.query_Z(x, bigram)
    prev_token = token
    prev_bigram = bigram
    prev_bigram_with_context = x

* I
* would
* like
* to
* go
* to
* the
* bar
  [147 162 194 254 436]
  the
  [ 41  94 147 162 436]
  the
  the
  the
  [147 162 262 335 436]
  the
  the
  [ 41  94 147 202 436]
  [ 41  94 147 162 436]
  the
  the
  the
  [147 162 262 335 436]
  the
  the
  [ 41  94 147 202 436]
  the
  [ 22  41 194 395 436]


In [15]:
R_list = list(map(lambda _: create_context_mem(), range(7)))
mem = TriadicMemory(N, P)

In [18]:
x, y, prediction = (my_empty_sdr(),) * 3

for run in range(1):
    hits, misses = 0, 0
    
    for word_ind, word in tqdm(enumerate(words), total=len(words)):
        token = codec(word)
        t_list = []
        
        for R in R_list:
            t = t_list[-1] if t_list else token
            t_list.append(R(t))

        if my_sdr_overlap(prediction, token) < P:
            mem.store(x, y, token)
            misses += 1
        else:
            hits += 1
    
        x = my_sdr_union(t_list[0], t_list[3])
        y = my_sdr_union(t_list[1], t_list[6])
        prediction = mem.query_Z(x, y)

    print(f'{hits / (hits + misses):.4f}, hits={hits}, misses={misses}')

  0%|          | 0/50000 [00:00<?, ?it/s]

0.8421, hits=42104, misses=7896


In [42]:
# prompt = preprocess_text("London is a captial of great britain")
prompt = preprocess_text("Where would you like to go tonight?")
# prompt = preprocess_text("Rusty key scraped round harshly twice and, when the heavy door had been")
# prompt = words[:50]
x, y = (my_empty_sdr(),) * 2

for word in prompt + [None] * 20:
    if word is None:
        # Prompt is over, now let's see what system generates in response to consumed prompt
        token = mem.query_Z(x, y)
        word = codec(token)

        if len(token) > P:
            for sdr, s in codec_cache.sdr_to_str.items():
                if my_sdr_overlap(sdr, token) == P:
                    word = s
                    token = sdr
                    break
        
        print(" ", token, word)
    else:
        token = codec(word)
        print("*", token, word)
        
    t_list = []
    
    for R in R_list:
        t = t_list[-1] if t_list else token
        t_list.append(R(t))

    x = my_sdr_union(t_list[0], t_list[3])
    y = my_sdr_union(t_list[1], t_list[6])

# for _ in range(20):
#     token = mem.query_Z(x, y)
#     print(codec(token))

#     t_list = []
    
#     for R in R_list:
#         t = t_list[-1] if t_list else token
#         t_list.append(R(t))

#     x = my_sdr_union(t_list[0], t_list[3])
#     y = my_sdr_union(t_list[1], t_list[6])    

* [ 44  51 312 333 489] Where
* [ 31 217 392 420 438] would
* [ 55 123 180 426 476] you
* [150 168 265 270 428] like
* [ 13  80 304 447 478] to
* [ 84 161 383 395 404] go
* [181 250 319 322 385] tonight
  [ 41  94 147 162 202] [ 41  94 147 162 202]
  [ 41  94 147 162 202] [ 41  94 147 162 202]
  [ 41  94 147 162 424 436] [ 41  94 147 162 424 436]
  [ 41  94 147 202 436] [ 41  94 147 202 436]
  [ 41  94 147 162 436] [ 41  94 147 162 436]
  [ 41  94 147 162 424 436] [ 41  94 147 162 424 436]
  [ 41 147 194 321 436] [ 41 147 194 321 436]
  [ 41 138 147 217 436] [ 41 138 147 217 436]
  [ 41  94 147 162 202] [ 41  94 147 162 202]
  [ 41  94 147 162 202] [ 41  94 147 162 202]
  [ 41  94 147 217 436] [ 41  94 147 217 436]
  [ 41  94 147 162 202] [ 41  94 147 162 202]
  [ 41  94 147 162 202 436] [ 41  94 147 162 202 436]
  [ 41  94 147 162 436] [ 41  94 147 162 436]
  [ 41 147 162 202 373] [ 41 147 162 202 373]
  [ 41  94 147 162 436] [ 41  94 147 162 436]
  [ 41  94 147 162 436] [ 41  94 147 

In [44]:
codec("Where")

array([ 42,  94, 271, 274, 297])

In [45]:
codec("the")

array([138, 243, 264, 435, 445])